# LAPTOP DATA CLEANING

## Business Understanding

Laptop prices vary depending on their specifications, features, and brands. Understanding the factors associated with laptop prices can help consumers make informed purchasing decisions.

### Business Question

What factors influence laptop prices in the global market?

In [115]:
# import necessary dependancies
import pandas as pd
import opendatasets as od
import os
import re

## Data understanding

In [116]:
data_url="https://www.kaggle.com/datasets/mdriazhossain/global-laptop-selling-dataset"

# download data
od.download(data_url,"../data")



Skipping, found downloaded files in "../data/global-laptop-selling-dataset" (use force=True to force download)


In [117]:
#list downloaded files
os.listdir("../data/global-laptop-selling-dataset")

['clean_global_laptop_selling_data.csv', 'global_laptop_selling_data.csv']

In [118]:
# load data
laptop_df=pd.read_csv("../data/global-laptop-selling-dataset/global_laptop_selling_data.csv",index_col="laptop_ID")


In [119]:
# display first 5 rows
laptop_df.head()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
laptop_ID,,,,,,,,,,,,
1,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 2.3GHz,8GB,128GB SSD,Intel Iris Plus Graphics 640,macOS,1.37kg,1339.69
2,Apple,Macbook Air,Ultrabook,13.3,1440x900,Intel Core i5 1.8GHz,8GB,128GB Flash Storage,Intel HD Graphics 6000,macOS,1.34kg,898.94
3,HP,250 G6,Notebook,15.6,Full HD 1920x1080,Intel Core i5 7200U 2.5GHz,8GB,256GB SSD,Intel HD Graphics 620,No OS,1.86kg,575.00
4,Apple,MacBook Pro,Ultrabook,15.4,IPS Panel Retina Display 2880x1800,Intel Core i7 2.7GHz,16GB,512GB SSD,AMD Radeon Pro 455,macOS,1.83kg,2537.45
5,Apple,MacBook Pro,Ultrabook,13.3,IPS Panel Retina Display 2560x1600,Intel Core i5 3.1GHz,8GB,256GB SSD,Intel Iris Plus Graphics 650,macOS,1.37kg,1803.60


In [120]:
# display last 5 rows
laptop_df.tail()

,Company,Product,TypeName,Inches,ScreenResolution,Cpu,Ram,Memory,Gpu,OpSys,Weight,Price_euros
laptop_ID,,,,,,,,,,,,
1316,Lenovo,Yoga 500-14ISK,2 in 1 Convertible,14.0,IPS Panel Full HD / Touchscreen 1920x1080,Intel Core i7 6500U 2.5GHz,4GB,128GB SSD,Intel HD Graphics 520,Windows 10,1.8kg,638.0
1317,Lenovo,Yoga 900-13ISK,2 in 1 Convertible,13.3,IPS Panel Quad HD+ / Touchscreen 3200x1800,Intel Core i7 6500U 2.5GHz,16GB,512GB SSD,Intel HD Graphics 520,Windows 10,1.3kg,1499.0
1318,Lenovo,IdeaPad 100S-14IBR,Notebook,14.0,1366x768,Intel Celeron Dual Core N3050 1.6GHz,2GB,64GB Flash Storage,Intel HD Graphics,Windows 10,1.5kg,229.0
1319,HP,15-AC110nv (i7-6500U/6GB/1TB/Radeon,Notebook,15.6,1366x768,Intel Core i7 6500U 2.5GHz,6GB,1TB HDD,AMD Radeon R5 M330,Windows 10,2.19kg,764.0
1320,Asus,X553SA-XX031T (N3050/4GB/500GB/W10),Notebook,15.6,1366x768,Intel Celeron Dual Core N3050 1.6GHz,4GB,500GB HDD,Intel HD Graphics,Windows 10,2.2kg,369.0


In [121]:
# dispay number of rows and columns
laptop_df.shape

(1303, 12)

The data has 1303 rows and 13 columns

In [122]:
# get general summary of the data
laptop_df.info()

<class 'pandas.DataFrame'>
Index: 1303 entries, 1 to 1320
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   str    
 1   Product           1303 non-null   str    
 2   TypeName          1303 non-null   str    
 3   Inches            1303 non-null   float64
 4   ScreenResolution  1303 non-null   str    
 5   Cpu               1303 non-null   str    
 6   Ram               1303 non-null   str    
 7   Memory            1303 non-null   str    
 8   Gpu               1303 non-null   str    
 9   OpSys             1303 non-null   str    
 10  Weight            1303 non-null   str    
 11  Price_euros       1303 non-null   float64
dtypes: float64(2), str(10)
memory usage: 132.3 KB


Their are no missing values

In [123]:
# check for duplicates
laptop_df.duplicated().sum()

np.int64(28)

In [124]:
# list unique values in product
laptop_df["Product"].unique()

<StringArray>
[                              'MacBook Pro',
                               'Macbook Air',
                                    '250 G6',
                                  'Aspire 3',
                           'ZenBook UX430UN',
                                   'Swift 3',
                             'Inspiron 3567',
                               'MacBook 12"',
                         'IdeaPad 320-15IKB',
                                    'XPS 13',
 ...
                               'GE62 Apache',
                            'Yoga 500-14IBD',
                      'ZenBook UX305CA-UBM1',
                            'Aspire ES1-531',
                       'Pavilion 15-AW003nv',
                          'Stream 11-Y000na',
 'X556UJ-XO044T (i7-6500U/4GB/500GB/GeForce',
                            'Yoga 500-14ISK',
       '15-AC110nv (i7-6500U/6GB/1TB/Radeon',
       'X553SA-XX031T (N3050/4GB/500GB/W10)']
Length: 618, dtype: str

The data has no duplicates

### Cleaning Tasks

1. **ScreenResolution:** Extract the screen resolution into separate `ResolutionWidth` and `ResolutionHeight` columns and convert both to integers. Retain the `DisplayFeatures` in a separate column.

2. **RAM:** Remove the unit (GB) and convert the RAM capacity to an integer.

3. **Memory:** Separate storage capacity from storage type into distinct columns. Convert the storage capacity to a numeric format.

4. **CPU:** Extract the CPU brand, model, and clock speed into separate columns. Convert clock speed to a numeric data type.

5. **GPU:** Extract the GPU brand,  and model into separate columns .

6. **Weight:** Remove the unit (kg) and convert weight to a float.
7. **Product**: Remove redundant hardware specifications embedded in product names while preserving the actual laptop model names.
8. Drop  **ScreenResolution:** after separating `ResolutionWidth` and `ResolutionHeight` and `DisplayFeatures`
9. Drop `Memory ` after separating storage capacity and storage type
10. Drop `Cpu` after separating cpu brand,model and clockspeed
11. Drop `Gpu` after separating gpu brand and model
12. Create `OsFamily` column  that group operating system in broader family
13. Combine `ResolutionWidth` and `ResolutionWidth` into `Pixels` because they have multicolinerity 

## Cleaning

In [125]:
"""
    Extract and calculate the total storage capacity for a given storage type.
    Converts TB to GB and returns the total capacity in GB.
"""
def get_storage_capacity(memory,storage_type):
    if pd.isna(memory):
        return 0
    pattern=rf"(\d+(?:\.\d+)?)\s*(GB|TB)\s*{storage_type}"
    matches=re.findall(pattern,memory)
    total=0
    for capacity,unit in matches:
       
        capacity=float(capacity)
        
        if unit =="TB":
            capacity *=1024
            
        total+=capacity
        
    return total

In [126]:
cleaned_df=( 
 laptop_df.assign(
     Product=lambda x:x["Product"].str.replace(r"\(.*","",regex=True).str.strip(),
     ResolutionWidth=lambda x: x["ScreenResolution"].str.extract(r"(\d+)(?=[*x])").astype("int64"),
     ResolutionHeight=lambda x:x["ScreenResolution"].str.extract(r"(?<=[*x])(\d+)").astype("int64"),
     PixelsMP=lambda x: ((x["ResolutionWidth"] * x["ResolutionHeight"])/1000000).round(2),
     DisplayFeatures=lambda x:x["ScreenResolution"].str.extract(r"^(.*?)\s*\d+[*x]\d+"),
     CpuBrand=lambda x:x["Cpu"].str.extract(r"^(\w+)"),
     CpuModel=lambda x:x["Cpu"].str.extract(r"^\w+\s*(.+?)\s*\d+(?:\.\d+)?\s*GHz"),
     ClockSpeed=lambda x:x["Cpu"].str.extract(r"(\d+(?:\.\d+)?)GHz").astype(float),
     Ram=lambda x:x["Ram"].str.replace("GB","").astype(int),
     SSDCapacityGB=lambda x: x["Memory"].apply(lambda x:get_storage_capacity(x,"SSD")),
     HDDCapacityGB=lambda x: x["Memory"].apply(lambda x:get_storage_capacity(x,"HDD")),
     FlashStorageGB=lambda x:x["Memory"].apply(lambda x:get_storage_capacity(x,"Flash Storage")),
     HybridStorageGB=lambda x: x["Memory"].apply(lambda x:get_storage_capacity(x,"Hybrid")),
     WeightKg=lambda x:x["Weight"].str.replace("kg","").astype(float),
     GpuBrand=lambda x: x["Gpu"].str.split(" ",expand=True)[0],
     GpuModel=lambda x: x["Gpu"].str.split(" ",expand=True)[[1,2,3,4]].fillna("").astype(str).agg(" ".join,axis=1).str.strip(),
     OsFamily=lambda x:x["OpSys"].replace({
"Windows 10" :"Windows",              
"Windows 7":"Windows",       
"Mac OS X": "macOS",   
"Windows 10 S": "Windows"
         })
).drop(columns=["ScreenResolution","Memory","Weight","Cpu","Gpu"])
   
)

### Reinspect the Data to Verify the Cleaning

In [127]:
#display first 10 rows
cleaned_df.head(10)




,Company,Product,TypeName,Inches,Ram,OpSys,Price_euros,ResolutionWidth,ResolutionHeight,PixelsMP,...,CpuModel,ClockSpeed,SSDCapacityGB,HDDCapacityGB,FlashStorageGB,HybridStorageGB,WeightKg,GpuBrand,GpuModel,OsFamily
laptop_ID,,,,,,,,,,,,,,,,,,,,,
1,Apple,MacBook Pro,Ultrabook,13.3,8,macOS,1339.69,2560,1600,4.10,...,Core i5,2.3,128.0,0.0,0.0,0.0,1.37,Intel,Iris Plus Graphics 640,macOS
2,Apple,Macbook Air,Ultrabook,13.3,8,macOS,898.94,1440,900,1.30,...,Core i5,1.8,0.0,0.0,128.0,0.0,1.34,Intel,HD Graphics 6000,macOS
3,HP,250 G6,Notebook,15.6,8,No OS,575.00,1920,1080,2.07,...,Core i5 7200U,2.5,256.0,0.0,0.0,0.0,1.86,Intel,HD Graphics 620,No OS
4,Apple,MacBook Pro,Ultrabook,15.4,16,macOS,2537.45,2880,1800,5.18,...,Core i7,2.7,512.0,0.0,0.0,0.0,1.83,AMD,Radeon Pro 455,macOS
5,Apple,MacBook Pro,Ultrabook,13.3,8,macOS,1803.60,2560,1600,4.10,...,Core i5,3.1,256.0,0.0,0.0,0.0,1.37,Intel,Iris Plus Graphics 650,macOS
6,Acer,Aspire 3,Notebook,15.6,4,Windows 10,400.00,1366,768,1.05,...,A9-Series 9420,3.0,0.0,500.0,0.0,0.0,2.10,AMD,Radeon R5,Windows
7,Apple,MacBook Pro,Ultrabook,15.4,16,Mac OS X,2139.97,2880,1800,5.18,...,Core i7,2.2,0.0,0.0,256.0,0.0,2.04,Intel,Iris Pro Graphics,macOS
8,Apple,Macbook Air,Ultrabook,13.3,8,macOS,1158.70,1440,900,1.30,...,Core i5,1.8,0.0,0.0,256.0,0.0,1.34,Intel,HD Graphics 6000,macOS
9,Asus,ZenBook UX430UN,Ultrabook,14.0,16,Windows 10,1495.00,1920,1080,2.07,...,Core i7 8550U,1.8,512.0,0.0,0.0,0.0,1.30,Nvidia,GeForce MX150,Windows


In [128]:
# display number of rows and columns
cleaned_df.shape

(1303, 22)

There are 1303 rows and columns

In [129]:
# get general summary of your data
cleaned_df.info()

<class 'pandas.DataFrame'>
Index: 1303 entries, 1 to 1320
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Company           1303 non-null   str    
 1   Product           1303 non-null   str    
 2   TypeName          1303 non-null   str    
 3   Inches            1303 non-null   float64
 4   Ram               1303 non-null   int64  
 5   OpSys             1303 non-null   str    
 6   Price_euros       1303 non-null   float64
 7   ResolutionWidth   1303 non-null   int64  
 8   ResolutionHeight  1303 non-null   int64  
 9   PixelsMP          1303 non-null   float64
 10  DisplayFeatures   1303 non-null   str    
 11  CpuBrand          1303 non-null   str    
 12  CpuModel          1303 non-null   str    
 13  ClockSpeed        1303 non-null   float64
 14  SSDCapacityGB     1303 non-null   float64
 15  HDDCapacityGB     1303 non-null   float64
 16  FlashStorageGB    1303 non-null   float64
 17  HybridStora

In [130]:
## check for duplicates
cleaned_df.duplicated().sum()

np.int64(28)

In [131]:
## save cleaned data
cleaned_df.to_csv("../data/global-laptop-selling-dataset/clean_global_laptop_selling_data.csv")